# B1 / B4 — Tokenisation Robustness and Short-Text Entropy/MTLD Stability

**Reviewer concerns addressed:**
- **B1** — *"Whitespace tokenisation conflates snake_case/camelCase identifiers, inflating code-token overlap and distorting readability scores. Re-run overlap and entropy metrics after identifier splitting and stopword filtering."*
- **B4** — *"Median doc length is 11–12 tokens. Add a note or supplementary histogram showing that entropy and MTLD differences are not artefacts of 3–5 token documents."*

This notebook:
1. **Identifier splitting** — camelCase and snake_case → constituent words; re-runs entropy, overlap, and MTLD.
2. **Stopword filtering** — removes standard English stopwords after splitting; re-runs same metrics.
3. **Short-text stability check** — filters to docs with ≥ 15 tokens; re-runs entropy and MTLD.
4. Produces a unified effect-size comparison table across all four conditions.
5. Saves to `revision_outputs/tokenisation_robustness.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

sns.set_theme(style='whitegrid')
PALETTE = {'Agentic': '#A7C7E7', 'Developer': '#BDE5B8'}

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset', 'data', 'updated_dataset_metrics.csv')
OUT_DIR   = os.path.join(os.getcwd(), 'revision_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
for col in ['doc_lines', 'doc_entropy', 'doc_code_overlap', 'doc_redundancy']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.dropna(subset=['doc_entropy', 'doc_code_overlap', 'doc_redundancy']).copy()
df = df[df['doc_lines'] > 0].copy()
print(f"Documented-function dataset: {df.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOKENIZER DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────

def tokenize_original(text):
    """Original tokenizer from existing notebooks — alphabetic runs."""
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text.lower())

def split_identifier(token):
    """Split a single camelCase or snake_case identifier into words."""
    # snake_case: split on underscores
    parts = token.split('_')
    result = []
    for part in parts:
        if not part:
            continue
        # camelCase: split before uppercase letters that follow lowercase
        sub = re.sub(r'([a-z])([A-Z])', r'\1 \2', part)
        # Handle sequences like 'XMLParser' -> 'XML Parser'
        sub = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', sub)
        result.extend(sub.lower().split())
    return [w for w in result if w]

def tokenize_split(text):
    """Tokenize then split camelCase/snake_case identifiers into words."""
    if not isinstance(text, str):
        return []
    raw_tokens = re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text)
    words = []
    for tok in raw_tokens:
        words.extend(split_identifier(tok))
    return [w for w in words if len(w) > 1]  # drop single-char residuals

# Test the splitter
print('Identifier splitting examples:')
for ex in ['getDocumentLength', 'doc_token_count', 'XMLParser', 'my_function_name', 'CamelCaseVar']:
    print(f'  {ex!r:30s} -> {split_identifier(ex)}')

In [ ]:
# NLTK stopwords
try:
    from nltk.corpus import stopwords
    import nltk
    nltk.download('stopwords', quiet=True)
    STOPWORDS = set(stopwords.words('english'))
    print(f'Loaded {len(STOPWORDS)} English stopwords')
except ImportError:
    # Fallback minimal set if nltk not available
    STOPWORDS = {
        'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
        'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'could', 'should', 'may', 'might', 'can', 'that', 'this', 'these',
        'those', 'it', 'its', 'if', 'as', 'not', 'no'
    }
    print(f'NLTK not available — using {len(STOPWORDS)}-word fallback stoplist')

def tokenize_split_nostop(text):
    """Identifier-split tokenizer with English stopword removal."""
    return [w for w in tokenize_split(text) if w not in STOPWORDS]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# METRIC COMPUTATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def shannon_entropy(tokens):
    """Shannon entropy (bits) of a token list."""
    if len(tokens) == 0:
        return np.nan
    counts = pd.Series(tokens).value_counts()
    probs  = counts / counts.sum()
    return -(probs * np.log2(probs)).sum()

def code_overlap(doc_tokens, func_tokens):
    """Fraction of doc tokens that also appear in the function code."""
    if not doc_tokens:
        return np.nan
    func_set = set(func_tokens)
    return sum(1 for t in doc_tokens if t in func_set) / len(doc_tokens)

def mw_effect(a, b):
    """Mann-Whitney U test + rank-biserial r for two Series."""
    a = a.dropna().values
    b = b.dropna().values
    if len(a) < 5 or len(b) < 5:
        return np.nan, np.nan, np.nan
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    r = 1 - (2 * stat) / (len(a) * len(b))
    return stat, p, r

def interpret_r(r):
    ar = abs(r)
    if ar < 0.1: return 'negligible'
    if ar < 0.3: return 'small'
    if ar < 0.5: return 'medium'
    return 'large'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILD TOKEN LISTS FOR EACH CONDITION
# ─────────────────────────────────────────────────────────────────────────────
print('Building token lists (this may take 1-2 minutes)...')

df_work = df[['doc_text', 'function', 'group', 'doc_code_overlap', 'doc_entropy']].copy()

# Original
df_work['tok_orig']   = df_work['doc_text'].apply(tokenize_original)
df_work['func_orig']  = df_work['function'].apply(tokenize_original)

# Identifier-split
df_work['tok_split']  = df_work['doc_text'].apply(tokenize_split)
df_work['func_split'] = df_work['function'].apply(tokenize_split)

# Identifier-split + stopwords removed
df_work['tok_nostop'] = df_work['doc_text'].apply(tokenize_split_nostop)
df_work['func_nostop']= df_work['function'].apply(tokenize_split_nostop)

# Compute metrics for each condition
for cond in ['orig', 'split', 'nostop']:
    df_work[f'entropy_{cond}'] = df_work[f'tok_{cond}'].apply(shannon_entropy)
    df_work[f'overlap_{cond}'] = df_work.apply(
        lambda r: code_overlap(r[f'tok_{cond}'], r[f'func_{cond}']), axis=1
    )
    df_work[f'n_tok_{cond}']   = df_work[f'tok_{cond}'].apply(len)

print('Done.')
print(f"Median token counts:")
for cond in ['orig', 'split', 'nostop']:
    print(f"  {cond:8s}: {df_work[f'n_tok_{cond}'].median():.1f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MTLD — compute for all conditions
# ─────────────────────────────────────────────────────────────────────────────
try:
    from lexicalrichness import LexicalRichness
    HAS_LR = True
except ImportError:
    HAS_LR = False
    print('lexicalrichness not installed — MTLD will be skipped')

def get_mtld(tokens):
    if not HAS_LR or len(tokens) < 10:
        return np.nan
    try:
        lex = LexicalRichness(' '.join(tokens))
        return lex.mtld(threshold=0.72)
    except Exception:
        return np.nan

if HAS_LR:
    print('Computing MTLD (may take a few minutes)...')
    for cond in ['orig', 'split', 'nostop']:
        df_work[f'mtld_{cond}'] = df_work[f'tok_{cond}'].apply(get_mtld)
    print('MTLD done.')
else:
    for cond in ['orig', 'split', 'nostop']:
        df_work[f'mtld_{cond}'] = np.nan

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SHORT-TEXT STABILITY: filter to ≥15 tokens (original tokenizer)
# ─────────────────────────────────────────────────────────────────────────────
MIN_TOKENS = 15
df_long = df_work[df_work['n_tok_orig'] >= MIN_TOKENS].copy()

n_total = len(df_work)
n_long  = len(df_long)
print(f"\nShort-text stability check:")
print(f"  Full dataset:     {n_total} docs")
print(f"  ≥{MIN_TOKENS} tokens:    {n_long} docs ({100*n_long/n_total:.1f}%)")
print(f"  Excluded:         {n_total - n_long} docs ({100*(1 - n_long/n_total):.1f}%)")

if HAS_LR:
    df_long['mtld_long'] = df_long['tok_orig'].apply(get_mtld)
else:
    df_long['mtld_long'] = np.nan

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EFFECT SIZE TABLE — compare across all four conditions
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== Effect Size Comparison Table ===')

conditions = [
    ('original',              df_work,  'entropy_orig',  'overlap_orig',  'mtld_orig'),
    ('identifier-split',      df_work,  'entropy_split', 'overlap_split', 'mtld_split'),
    ('stopword-filtered',     df_work,  'entropy_nostop','overlap_nostop','mtld_nostop'),
    (f'≥{MIN_TOKENS}-token subsample', df_long, 'entropy_orig',  'overlap_orig',  'mtld_long'),
]

rows = []
for cond_name, cond_df, ent_col, ovr_col, mtld_col in conditions:
    ag = cond_df[cond_df['group'] == 'agent']
    hu = cond_df[cond_df['group'] == 'human']

    _, p_ent, r_ent   = mw_effect(ag[ent_col], hu[ent_col])
    _, p_ovr, r_ovr   = mw_effect(ag[ovr_col], hu[ovr_col])
    _, p_mtld, r_mtld = mw_effect(ag[mtld_col], hu[mtld_col])

    rows.append({
        'condition':          cond_name,
        'n_agent':            len(ag),
        'n_developer':        len(hu),
        # Entropy
        'entropy_r_rb':       r_ent,
        'entropy_p':          p_ent,
        'entropy_sig':        (p_ent is not np.nan) and (p_ent < 0.05),
        'entropy_effect':     interpret_r(r_ent) if r_ent is not np.nan else 'n/a',
        # Overlap
        'overlap_r_rb':       r_ovr,
        'overlap_p':          p_ovr,
        'overlap_sig':        (p_ovr is not np.nan) and (p_ovr < 0.05),
        'overlap_effect':     interpret_r(r_ovr) if r_ovr is not np.nan else 'n/a',
        # MTLD
        'mtld_r_rb':          r_mtld,
        'mtld_p':             p_mtld,
        'mtld_sig':           (p_mtld is not np.nan) and (p_mtld < 0.05),
        'mtld_effect':        interpret_r(r_mtld) if r_mtld is not np.nan else 'n/a',
    })

    print(f"\n[{cond_name}] n_agent={len(ag)}, n_dev={len(hu)}")
    print(f"  entropy:  r={r_ent:.4f} {interpret_r(r_ent):10s}  p={p_ent:.3e}")
    print(f"  overlap:  r={r_ovr:.4f} {interpret_r(r_ovr):10s}  p={p_ovr:.3e}")
    if r_mtld is not np.nan and not np.isnan(r_mtld):
        print(f"  MTLD:     r={r_mtld:.4f} {interpret_r(r_mtld):10s}  p={p_mtld:.3e}")

robustness_df = pd.DataFrame(rows)
display(robustness_df[['condition', 'n_agent', 'n_developer',
                        'entropy_r_rb', 'entropy_effect',
                        'overlap_r_rb', 'overlap_effect',
                        'mtld_r_rb',    'mtld_effect']].to_string(index=False, float_format='{:.4f}'.format))

robustness_df.to_csv(os.path.join(OUT_DIR, 'tokenisation_robustness.csv'), index=False)
print('\nSaved tokenisation_robustness.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SUPPLEMENTARY: token-length histogram (B4 — short-text distribution)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for ax, grp, color, label in [
    (axes[0], 'agent',  '#A7C7E7', 'Agentic'),
    (axes[1], 'human',  '#BDE5B8', 'Developer')
]:
    lengths = df_work[df_work['group'] == grp]['n_tok_orig']
    # Clip for readability
    clipped = lengths.clip(upper=100)
    ax.hist(clipped, bins=40, color=color, edgecolor='grey', linewidth=0.5)
    ax.axvline(lengths.median(), color='red', linestyle='--', label=f'Median={lengths.median():.0f}')
    ax.axvline(MIN_TOKENS, color='black', linestyle=':', label=f'≥{MIN_TOKENS}-token cutoff')
    ax.set_title(f'{label} documentation token length', fontsize=14, fontweight='bold')
    ax.set_xlabel('Token count (clipped at 100)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'token_length_histogram.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved token_length_histogram.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERPRETATION: flag whether effect sizes change materially (>0.05 difference)
# ─────────────────────────────────────────────────────────────────────────────
MATERIAL_THRESHOLD = 0.05

orig_row = robustness_df[robustness_df['condition'] == 'original'].iloc[0]

print('=== Sensitivity Summary ===')
print(f"Threshold for material change in rank-biserial r: >{MATERIAL_THRESHOLD}")

for _, row in robustness_df[robustness_df['condition'] != 'original'].iterrows():
    for metric in ['entropy', 'overlap', 'mtld']:
        orig_r = orig_row[f'{metric}_r_rb']
        new_r  = row[f'{metric}_r_rb']
        if pd.isna(orig_r) or pd.isna(new_r):
            continue
        diff = abs(new_r - orig_r)
        flag = ' *** MATERIAL CHANGE' if diff > MATERIAL_THRESHOLD else ''
        print(f"  [{row['condition']}] {metric}: {orig_r:.4f} -> {new_r:.4f}  Δ={diff:.4f}{flag}")

## Revision note

- **What this adds:** Demonstrates that the entropy and code-overlap findings are not artefacts of whitespace tokenisation treating compound identifiers as single tokens, nor are they driven by very short (3–5 token) documents. If effect sizes are stable (Δ < 0.05) across conditions, this strongly defends the original tokenisation choice.
- **Where to cite in paper:** Add a paragraph to Section 3.3 (Tokenisation) and a supplementary table/figure. Reference the token-length histogram as Figure A1 in an appendix to address concern B4 directly.
- **What to watch for:** If overlap effect size drops substantially after identifier splitting (because many 'overlapping' tokens were compound identifiers that split into common words appearing in both doc and code), revise the overlap interpretation accordingly.